In [1]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from xgboost import XGBRegressor

In [11]:
df = pd.read_excel("Tension.xlsx")
df = df.drop(columns=['Mixture', 'Specimen'])
df = df.dropna(subset=['Fiber Type'])
df = df.reset_index(drop=True)

In [12]:
df['has_Silica_Fume'] = (df['Silica Fume'] > 0).astype(int)

df['FlyAshF_x_SilicaFume'] = df['Fly ash F'] * df['Silica Fume']
df['Length_x_SilicaFume']   = df['Length (mm)'] * df['Silica Fume']
df['FiberVol_x_Length']     = df['Fiber Volume'] * df['Length (mm)']
df['FlyAshF_x_WaterReducer'] = df['Fly ash F'] * df['Water Reducer / SP']

df['Total_SCM']   = df['Fly ash C'] + df['Fly ash F'] + df['GGBS'] + df['Silica Fume']
df['SCM_ratio']   = df['Total_SCM'] / df['Cement']
df['Total_paste'] = df['Cement'] + df['Water'] + df['Total_SCM']
df['Water_Binder'] = df['Water'] / (df['Cement'] + df['Total_SCM'])

df_original = df.copy()
df = pd.get_dummies(df, columns=['Fiber Type'], drop_first=False)

print(f"Final shape: {df.shape}")

Final shape: (503, 53)


In [13]:
X = df.drop(columns=['Second Stress', 'Second Strain'])
y_stress = df['Second Stress']
y_strain = df['Second Strain']

In [14]:
X_train, X_test, \
y_stress_train, y_stress_test, \
y_strain_train, y_strain_test = train_test_split(
    X, y_stress, y_strain,
    test_size=0.20,
    random_state=42
)

In [15]:
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X.columns, index=X_train.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X.columns, index=X_test.index
)

kf = KFold(n_splits=5, shuffle=True, random_state=42)
print(f"Train: {X_train_scaled.shape[0]} rows")
print(f"Test:  {X_test_scaled.shape[0]} rows")
print(f"Features: {X_train_scaled.shape[1]}")


Train: 402 rows
Test:  101 rows
Features: 51


In [16]:
print("\n" + "="*50)
print("  STRESS BASELINES")
print("="*50)

stress_models = {
    "RF":  RandomForestRegressor(random_state=42),
    "XGB": XGBRegressor(random_state=42, objective='reg:squarederror', verbosity=0),
}

stress_results = []
for name, model in stress_models.items():
    cv = cross_validate(model, X_train_scaled, y_stress_train, cv=kf,
                        scoring=('r2', 'neg_root_mean_squared_error'))
    r2   = cv['test_r2'].mean()
    rmse = -cv['test_neg_root_mean_squared_error'].mean()
    stress_results.append({"Model": name, "CV R2": round(r2, 4), "CV RMSE": round(rmse, 4)})
    print(f"  {name:5s}  R²={r2:.4f}  RMSE={rmse:.4f}")



  STRESS BASELINES
  RF     R²=0.8215  RMSE=0.6408
  XGB    R²=0.8351  RMSE=0.6120


In [17]:
print("\n" + "="*50)
print("  STRAIN BASELINES")
print("="*50)

strain_models = {
    "RF":  RandomForestRegressor(random_state=42),
    "XGB": XGBRegressor(random_state=42, objective='reg:squarederror', verbosity=0),
}

strain_results = []
for name, model in strain_models.items():
    cv = cross_validate(model, X_train_scaled, y_strain_train, cv=kf,
                        scoring=('r2', 'neg_root_mean_squared_error'))
    r2   = cv['test_r2'].mean()
    rmse = -cv['test_neg_root_mean_squared_error'].mean()
    strain_results.append({"Model": name, "CV R2": round(r2, 4), "CV RMSE": round(rmse, 4)})
    print(f"  {name:5s}  R²={r2:.4f}  RMSE={rmse:.4f}")


  STRAIN BASELINES
  RF     R²=0.4597  RMSE=0.0139
  XGB    R²=0.4181  RMSE=0.0144


In [10]:
print("\n" + "="*50)
print("  TEST SET RESULTS")
print("="*50)

def test_eval(model, X_tr, y_tr, X_te, y_te, name, target):
    model.fit(X_tr, y_tr)
    pred = model.predict(X_te)
    r2   = r2_score(y_te, pred)
    rmse = np.sqrt(mean_squared_error(y_te, pred))
    mae  = mean_absolute_error(y_te, pred)
    print(f"  {name:5s} → {target:15s}  R²={r2:.4f}  RMSE={rmse:.4f}  MAE={mae:.4f}")
    return pred

print("\nStress:")
for name, model in stress_models.items():
    test_eval(model, X_train_scaled, y_stress_train,
              X_test_scaled, y_stress_test, name, "Stress")

print("\nStrain:")
for name, model in strain_models.items():
    test_eval(model, X_train_scaled, y_strain_train,
              X_test_scaled, y_strain_test, name, "Strain")

print("\n✅ Baselines complete.")


  TEST SET RESULTS

Stress:
  RF    → Stress           R²=0.8711  RMSE=0.6641  MAE=0.4711
  XGB   → Stress           R²=0.8365  RMSE=0.7480  MAE=0.5049

Strain:
  RF    → Strain           R²=0.7450  RMSE=0.0119  MAE=0.0089
  XGB   → Strain           R²=0.7394  RMSE=0.0121  MAE=0.0090

✅ Baselines complete.
